# The $\alpha,\!\beta$-CROWN Verifier
## Sound Bounds and Satisfiability Verification

### **Tutorial page: [abcrown.org/tutorial](https://abcrown.org/tutorial)**

For a nonlinear function $f:\mathbb{R}^n\to\mathbb{R}^m$ and an input set
$\mathcal{X}$, the verifier can compute sound bounds

$$
\underline y_i\leq f_i(x)\leq\overline y_i
\qquad \forall x\in\mathcal{X},
$$

and prove output properties

$$
f(x)\in\mathcal{Y}_{\mathrm{safe}}
\qquad \forall x\in\mathcal{X}.
$$

## Setup
### Colab GPU Environment

Select **Runtime > Change runtime type > GPU** before executing the
installation cell.

<p align="center">
  <img
    src="https://github.com/Verified-Intelligence/abCROWN_Control_Tutorial/blob/main/figures/colab_setup.png?raw=1"
    alt="Colab GPU runtime setup"
    width="48%"
  />
</p>

In [ ]:
%%capture
# Work from $HOME and remove any previous clone.
%cd $HOME
!rm -rf alpha-beta-CROWN

# Clone submodules together with the main repository.
!git clone --recursive https://github.com/Verified-Intelligence/alpha-beta-CROWN.git
%cd alpha-beta-CROWN
!uv pip install --system -e .

## Verifier Workflow

The verifier workflow has four steps.

1. **Implement the nonlinear function as an `nn.Module`.** Write the map to analyze as an
   `nn.Module`; `forward` uses `torch` operations and returns the tensor output.
2. **Create symbolic variables, solver settings, and the solver.**
   `input_vars(...)` and `output_vars(...)` define symbolic input and output
   vectors. `ConfigBuilder` stores solver settings. `ABCrownSolver` binds the
   model, symbolic variables, and solver settings together.
3. **Specify constraints with Python expressions.** `IOConstraints` stores
   input domains and output properties written with indexing, arithmetic,
   comparisons, parentheses, `&`, and `|`.
4. **Call the solver.** The solver provides calls such as `compute_bounds(...)`,
   `verify(...)`, `minimize(...)`, and `maximize(...)` for the requested task.


In [ ]:
#@title Helper functions { display-mode: "form" }
!uv pip install --system matplotlib > /dev/null

import contextlib
import functools
import logging
import os
import warnings

@contextlib.contextmanager
def _quiet_abcrown_output():
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield

with _quiet_abcrown_output(), warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    import abcrown as _abcrown

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

if not hasattr(_abcrown.ABCrownSolver, "_quiet_original_compute_bounds"):
    _abcrown.ABCrownSolver._quiet_original_compute_bounds = _abcrown.ABCrownSolver.compute_bounds
if not hasattr(_abcrown.ABCrownSolver, "_quiet_original_verify"):
    _abcrown.ABCrownSolver._quiet_original_verify = _abcrown.ABCrownSolver.verify
if not hasattr(_abcrown.IOConstraints, "_quiet_original_init"):
    _abcrown.IOConstraints._quiet_original_init = _abcrown.IOConstraints.__init__

@functools.wraps(_abcrown.ABCrownSolver._quiet_original_compute_bounds)
def _quiet_compute_bounds(self, *args, **kwargs):
    with _quiet_abcrown_output():
        return _abcrown.ABCrownSolver._quiet_original_compute_bounds(self, *args, **kwargs)

@functools.wraps(_abcrown.ABCrownSolver._quiet_original_verify)
def _quiet_verify(self, *args, **kwargs):
    with _quiet_abcrown_output():
        return _abcrown.ABCrownSolver._quiet_original_verify(self, *args, **kwargs)

@functools.wraps(_abcrown.IOConstraints._quiet_original_init)
def _quiet_ioconstraints_init(self, *args, **kwargs):
    with _quiet_abcrown_output():
        return _abcrown.IOConstraints._quiet_original_init(self, *args, **kwargs)

_abcrown.ABCrownSolver.compute_bounds = _quiet_compute_bounds
_abcrown.ABCrownSolver.verify = _quiet_verify
_abcrown.IOConstraints.__init__ = _quiet_ioconstraints_init

# Numerical and plotting utilities used by the figures.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
    "figure.titlesize": 20,
})
from matplotlib.patches import Rectangle

# Plotting helper functions only.
def as_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)

def vanderpol_step_numpy(states, dt=0.1, mu=1.0):
    q = states[..., 0]
    v = states[..., 1]
    q_next = q + dt * v
    v_next = v + dt * ((mu - q ** 2) * v - q)
    return np.stack((q_next, v_next), axis=-1)

def plot_vanderpol_trajectories():
    angles = np.linspace(0.0, 2.0 * np.pi, 18, endpoint=False)
    radii = np.array([0.55, 1.0, 1.45, 1.9])
    initial_states = np.array([
        [radius * np.cos(angle), radius * np.sin(angle)]
        for radius in radii
        for angle in angles
    ])
    steps = 170

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    colors = plt.get_cmap("viridis")(np.linspace(0.08, 0.92, len(initial_states)))
    for state0, color in zip(initial_states, colors):
        trajectory = np.empty((steps + 1, 2))
        trajectory[0] = state0
        for step in range(steps):
            trajectory[step + 1] = vanderpol_step_numpy(trajectory[step])
        ax.plot(
            trajectory[:, 0], trajectory[:, 1],
            color=color, alpha=0.76, linewidth=1.25,
        )

    ax.axhline(0.0, color="0.35", linewidth=0.8)
    ax.axvline(0.0, color="0.35", linewidth=0.8)
    ax.set(
        xlabel="$q$",
        ylabel="$v$",
        title="Van der Pol trajectories under one-step Euler dynamics",
    )
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

def sample_one_step(model, lower, upper):
    q = torch.linspace(float(lower[0]), float(upper[0]), 301, device=DEVICE)
    v = torch.linspace(float(lower[1]), float(upper[1]), 81, device=DEVICE)
    inputs = torch.cartesian_prod(q, v)
    with torch.no_grad():
        outputs = model(inputs)
    return inputs, outputs

def add_bound_box(ax, lower, upper, label, color, linestyle="-", linewidth=2.6):
    lower = as_numpy(lower)
    upper = as_numpy(upper)
    ax.add_patch(
        Rectangle(
            lower,
            upper[0] - lower[0],
            upper[1] - lower[1],
            fill=False,
            edgecolor=color,
            linestyle=linestyle,
            linewidth=linewidth,
            label=label,
        )
    )

def plot_reachable_box(outputs, bound_lower, bound_upper):
    outputs = outputs.detach().cpu()
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.scatter(
        outputs[::20, 0],
        outputs[::20, 1],
        s=12,
        color="tab:orange",
        alpha=0.55,
        label=r"sampled $f(\mathcal{X})$",
    )
    add_bound_box(ax, bound_lower, bound_upper, "output bound box", "tab:blue")
    ax.set(
        xlabel="$q^+$",
        ylabel="$v^+$",
        title="One-step reachable-set bound",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

def polygon_from_linear_bounds(directions, lower_bounds, upper_bounds, tol=1e-7):
    directions = np.asarray(directions, dtype=float)
    lower_bounds = as_numpy(lower_bounds).astype(float).reshape(-1)
    upper_bounds = as_numpy(upper_bounds).astype(float).reshape(-1)
    halfspace_A = np.concatenate((directions, -directions), axis=0)
    halfspace_b = np.concatenate((upper_bounds, -lower_bounds), axis=0)

    vertices = []
    for i in range(len(halfspace_A)):
        for j in range(i + 1, len(halfspace_A)):
            A_pair = np.stack((halfspace_A[i], halfspace_A[j]))
            if abs(np.linalg.det(A_pair)) < tol:
                continue
            candidate = np.linalg.solve(A_pair, np.array([halfspace_b[i], halfspace_b[j]]))
            if np.all(halfspace_A @ candidate <= halfspace_b + 1e-6):
                vertices.append(candidate)

    if not vertices:
        raise ValueError("Linear bounds do not form a bounded polygon.")
    vertices = np.unique(np.round(np.asarray(vertices), 10), axis=0)
    center = vertices.mean(axis=0)
    order = np.argsort(np.arctan2(vertices[:, 1] - center[1], vertices[:, 0] - center[0]))
    return vertices[order]

def direction_label(direction):
    a, b = direction
    if abs(a - 1.0) < 1e-9 and abs(b) < 1e-9:
        return r"$q^+$"
    if abs(a) < 1e-9 and abs(b - 1.0) < 1e-9:
        return r"$v^+$"
    if abs(a - 0.2) < 1e-9 and abs(b - 1.0) < 1e-9:
        return r"$0.2q^+ + v^+$"
    if abs(a + 0.2) < 1e-9 and abs(b - 1.0) < 1e-9:
        return r"$-0.2q^+ + v^+$"
    return rf"$({a:g}, {b:g})^\top y$"

def annotate_linear_bound_edges(ax, polygon, directions, lower_bounds, upper_bounds):
    directions = np.asarray(directions, dtype=float)
    lower_bounds = as_numpy(lower_bounds).astype(float).reshape(-1)
    upper_bounds = as_numpy(upper_bounds).astype(float).reshape(-1)
    center = polygon.mean(axis=0)
    x_span = ax.get_xlim()[1] - ax.get_xlim()[0]
    y_span = ax.get_ylim()[1] - ax.get_ylim()[0]

    for direction, lower, upper in zip(directions, lower_bounds, upper_bounds):
        values = polygon @ direction
        for bound_value, side in [(upper, "upper"), (lower, "lower")]:
            active = polygon[np.isclose(values, bound_value, atol=1e-5)]
            if len(active) < 2:
                continue
            midpoint = active.mean(axis=0)
            normal = direction / np.linalg.norm(direction)
            if np.dot(midpoint + normal * 0.01 - center, normal) < 0:
                normal = -normal
            text_xy = midpoint + normal * np.array([0.04 * x_span, 0.04 * y_span])
            ax.annotate(
                f"{direction_label(direction)} {side}",
                xy=midpoint,
                xytext=text_xy,
                ha="center",
                va="center",
                fontsize=13,
                color="tab:purple",
                arrowprops=dict(arrowstyle="-", color="tab:purple", lw=1.0, alpha=0.75),
                bbox=dict(boxstyle="round,pad=0.18", fc="white", ec="tab:purple", alpha=0.82),
            )

def plot_linear_reachable_set(outputs, directions, linear_lower, linear_upper):
    outputs = outputs.detach().cpu()
    polygon = polygon_from_linear_bounds(directions, linear_lower, linear_upper)

    fig, ax = plt.subplots(figsize=(9.5, 6.0))
    closed_polygon = np.vstack((polygon, polygon[0]))
    ax.fill(
        polygon[:, 0], polygon[:, 1],
        color="tab:purple", alpha=0.16, label="linear output bound",
    )
    ax.plot(
        closed_polygon[:, 0], closed_polygon[:, 1],
        color="tab:purple", linewidth=2.4,
    )
    ax.scatter(
        outputs[::20, 0], outputs[::20, 1],
        s=12, color="tab:orange", alpha=0.55, label=r"sampled $f(\mathcal{X})$",
    )

    all_points = np.vstack((polygon, outputs.numpy()))
    q_min, v_min = all_points.min(axis=0)
    q_max, v_max = all_points.max(axis=0)
    q_pad = 0.16 * (q_max - q_min)
    v_pad = 0.24 * (v_max - v_min)
    ax.set(
        xlim=(q_min - q_pad, q_max + q_pad),
        ylim=(v_min - v_pad, v_max + v_pad),
        xlabel="$q^+$",
        ylabel="$v^+$",
        title="Linear reachable-set bound",
    )
    annotate_linear_bound_edges(ax, polygon, directions, linear_lower, linear_upper)
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

def plot_verification_region(outputs, safe_lower, safe_upper):
    outputs = outputs.detach().cpu()
    safe_lower = as_numpy(safe_lower)
    safe_upper = as_numpy(safe_upper)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.add_patch(
        Rectangle(
            safe_lower,
            safe_upper[0] - safe_lower[0],
            safe_upper[1] - safe_lower[1],
            facecolor="tab:green",
            edgecolor="tab:green",
            alpha=0.14,
            linewidth=2.8,
            label="safe output set",
        )
    )
    ax.scatter(
        outputs[::20, 0], outputs[::20, 1],
        s=12, color="tab:orange", alpha=0.55, label=r"sampled $f(\mathcal{X})$",
    )
    all_points = np.vstack((outputs.numpy(), safe_lower, safe_upper))
    q_min, v_min = all_points.min(axis=0)
    q_max, v_max = all_points.max(axis=0)
    q_pad = 0.08 * (q_max - q_min)
    v_pad = 0.12 * (v_max - v_min)
    ax.set(
        xlim=(q_min - q_pad, q_max + q_pad),
        ylim=(v_min - v_pad, v_max + v_pad),
        xlabel="$q^+$",
        ylabel="$v^+$",
        title="Output set and safe set",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_verification_bounds(outputs, safe_lower, safe_upper):
    outputs = outputs.detach().cpu()
    initial_box = bound_results[0]
    bab_box = bound_results[-1]
    safe_lower = as_numpy(safe_lower)
    safe_upper = as_numpy(safe_upper)
    initial_lower = as_numpy(initial_box["lower"])
    initial_upper = as_numpy(initial_box["upper"])
    bab_lower = as_numpy(bab_box["lower"])
    bab_upper = as_numpy(bab_box["upper"])
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.add_patch(
        Rectangle(
            safe_lower,
            safe_upper[0] - safe_lower[0],
            safe_upper[1] - safe_lower[1],
            facecolor="tab:green",
            edgecolor="tab:green",
            alpha=0.12,
            linewidth=2.8,
            label="safe output set",
        )
    )
    ax.scatter(
        outputs[::20, 0], outputs[::20, 1],
        s=12, color="tab:orange", alpha=0.5, label=r"sampled $f(\mathcal{X})$",
    )
    add_bound_box(
        ax, initial_lower, initial_upper,
        "one-iteration CROWN bound", "0.35", "--", 3.0,
    )
    add_bound_box(
        ax, bab_lower, bab_upper,
        "Input BaB bound", "tab:blue", "-", 3.0,
    )
    all_points = np.vstack((
        outputs.numpy(),
        safe_lower,
        safe_upper,
        initial_lower,
        initial_upper,
        bab_lower,
        bab_upper,
    ))
    q_min, v_min = all_points.min(axis=0)
    q_max, v_max = all_points.max(axis=0)
    q_pad = 0.08 * (q_max - q_min)
    v_pad = 0.12 * (v_max - v_min)
    ax.set(
        xlim=(q_min - q_pad, q_max + q_pad),
        ylim=(v_min - v_pad, v_max + v_pad),
        xlabel="$q^+$",
        ylabel="$v^+$",
        title="CROWN and Input BaB output bounds",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

def affine_surfaces(model, lower, upper, lower_A, lower_bias, upper_A, upper_bias):
    q = torch.linspace(lower[0].cpu(), upper[0].cpu(), 81)
    v = torch.linspace(lower[1].cpu(), upper[1].cpu(), 41)
    Q, V = torch.meshgrid(q, v, indexing="ij")
    points = torch.stack((Q.reshape(-1), V.reshape(-1)), dim=1)
    with torch.no_grad():
        true = model(points.to(DEVICE))[:, 1].cpu().reshape(Q.shape)
    affine_lower = (points @ lower_A.T + lower_bias).reshape(Q.shape)
    affine_upper = (points @ upper_A.T + upper_bias).reshape(Q.shape)
    assert torch.all(affine_lower <= true + 1e-5)
    assert torch.all(true <= affine_upper + 1e-5)
    return Q, V, true, affine_lower, affine_upper

def draw_global_affine(ax, Q, V, true, affine_lower, affine_upper):
    ax.plot_surface(Q.numpy(), V.numpy(), true.numpy(), cmap="Greys", alpha=0.38, linewidth=0)
    ax.plot_surface(
        Q.numpy(), V.numpy(), affine_lower.numpy(),
        color="tab:blue", alpha=0.58, linewidth=0.2,
    )
    ax.plot_surface(
        Q.numpy(), V.numpy(), affine_upper.numpy(),
        color="tab:orange", alpha=0.52, linewidth=0.2,
    )
    ax.set(xlabel="$q$", ylabel="$v$", zlabel="$v^+$", title="Global affine bounds")
    ax.view_init(elev=26, azim=-60)

def plot_lower_convergence(iterations, lower_bounds, sampled_minimum):
    values = np.asarray(lower_bounds)
    margin = 0.06 * (sampled_minimum - values.min())
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    ax.plot(iterations, values, marker="o", linewidth=2.4, label=r"sound lower bound")
    ax.axhline(
        sampled_minimum,
        color="black",
        linestyle="--",
        linewidth=2,
        label="sampled minimum",
    )
    ax.set_ylim(values.min() - margin, sampled_minimum + margin)
    ax.set_xticks(iterations)
    ax.set(
        xlabel="Maximum BaB iterations",
        ylabel=r"Lower bound on $v^+$",
        title="Input BaB bound tightening",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_bound_comparison(outputs, first_lower, first_upper, final_lower, final_upper):
    outputs = outputs.detach().cpu()
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.scatter(
        outputs[::20, 0],
        outputs[::20, 1],
        s=12,
        color="tab:orange",
        alpha=0.5,
        label=r"sampled $f(\mathcal{X})$",
    )
    add_bound_box(ax, first_lower, first_upper, "1 iteration", "0.35", "--", 2.3)
    add_bound_box(ax, final_lower, final_upper, "8 iterations", "tab:blue", "-", 2.8)
    ax.set(
        xlabel="$q^+$",
        ylabel="$v^+$",
        title="Aggregate reachable-set bounds",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

def color_sequence(name, count):
    cmap = plt.get_cmap(name)
    if count <= 1:
        return [cmap(0.5)]
    return [cmap(value) for value in np.linspace(0.12, 0.90, count)]

def draw_local_planes(ax, records, colors, model, side):
    for record, color in zip(records, colors):
        q = torch.linspace(record["x_L"][0], record["x_U"][0], 17)
        v = torch.linspace(record["x_L"][1], record["x_U"][1], 11)
        Q, V = torch.meshgrid(q, v, indexing="ij")
        points = torch.stack((Q.reshape(-1), V.reshape(-1)), dim=1)
        affine = (points @ record["A"].T + record["bias"]).reshape(Q.shape)
        with torch.no_grad():
            true = model(points.to(DEVICE))[:, 1].cpu().reshape(Q.shape)
        if side == "lower":
            assert torch.all(affine <= true + 1e-5)
        else:
            assert torch.all(true <= affine + 1e-5)
        ax.plot_surface(
            Q.numpy(), V.numpy(), affine.numpy(),
            color=color, alpha=0.76, linewidth=0.25,
        )

def draw_piecewise_affine(ax, Q, V, true, lower_records, model):
    ax.plot_surface(Q.numpy(), V.numpy(), true.numpy(), cmap="Greys", alpha=0.18, linewidth=0)
    draw_local_planes(ax, lower_records, color_sequence("viridis", len(lower_records)), model, "lower")
    ax.set(
        xlabel="$q$",
        ylabel="$v$",
        zlabel="$v^+$",
        title=f"BaB subdomain lower affine bounds ({len(lower_records)} records)",
    )
    ax.view_init(elev=26, azim=-60)

def plot_linear_bound_comparison(
    model, lower, upper, lower_A, lower_bias, upper_A, upper_bias, lower_records
):
    Q, V, true, affine_lower, affine_upper = affine_surfaces(
        model, lower, upper, lower_A, lower_bias, upper_A, upper_bias
    )

    fig = plt.figure(figsize=(15, 5.8))
    left = fig.add_subplot(121, projection="3d")
    right = fig.add_subplot(122, projection="3d")
    draw_global_affine(left, Q, V, true, affine_lower, affine_upper)
    draw_piecewise_affine(right, Q, V, true, lower_records, model)
    plt.tight_layout()
    plt.show()

def plot_property(outputs, weak_lower, weak_upper, strong_lower, strong_upper, safe_lower, safe_upper):
    outputs = outputs.detach().cpu()
    safe_lower = as_numpy(safe_lower)
    safe_upper = as_numpy(safe_upper)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.add_patch(
        Rectangle(
            safe_lower,
            safe_upper[0] - safe_lower[0],
            safe_upper[1] - safe_lower[1],
            facecolor="tab:green",
            edgecolor="tab:green",
            alpha=0.12,
            linewidth=2.5,
            label="safe output set",
        )
    )
    ax.scatter(
        outputs[::20, 0], outputs[::20, 1],
        s=12, color="tab:orange", alpha=0.5, label=r"sampled $f(\mathcal{X})$",
    )
    add_bound_box(ax, weak_lower, weak_upper, "1 iteration", "0.35", "--", 2.2)
    add_bound_box(ax, strong_lower, strong_upper, "4 iterations", "tab:blue", "-", 2.8)
    ax.set(xlabel="$q^+$", ylabel="$v^+$", title="Output property and sound bounds")
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_counterexample(candidate_inputs, model, input_lower, input_upper, safe_lower, safe_upper):
    with torch.no_grad():
        candidate_outputs = model(candidate_inputs)

    counterexample_index = torch.argmin(candidate_outputs[:, 1])
    x_cex = as_numpy(candidate_inputs[counterexample_index])
    y_cex = as_numpy(candidate_outputs[counterexample_index])
    input_lower = as_numpy(input_lower)
    input_upper = as_numpy(input_upper)
    safe_lower = as_numpy(safe_lower)
    safe_upper = as_numpy(safe_upper)

    fig, ax = plt.subplots(figsize=(8.5, 5.4))
    ax.add_patch(
        Rectangle(
            input_lower,
            input_upper[0] - input_lower[0],
            input_upper[1] - input_lower[1],
            facecolor="0.65",
            edgecolor="0.25",
            alpha=0.28,
            linewidth=2.0,
            label=r"input set $\mathcal{X}$",
        )
    )
    ax.add_patch(
        Rectangle(
            safe_lower,
            safe_upper[0] - safe_lower[0],
            safe_upper[1] - safe_lower[1],
            facecolor="tab:green",
            edgecolor="tab:green",
            alpha=0.12,
            linewidth=2.5,
            label=r"strict output set $\mathcal{Y}_{\mathrm{strict}}$",
        )
    )
    ax.axhline(
        safe_lower[1],
        color="tab:red",
        linestyle="--",
        linewidth=2.0,
        label=r"lower boundary of $\mathcal{Y}_{\mathrm{strict}}$",
    )
    ax.scatter(
        x_cex[0], x_cex[1],
        marker="o", s=95, color="tab:blue", edgecolor="white", linewidth=0.8,
        zorder=5, label=r"counterexample input $x$",
    )
    ax.scatter(
        y_cex[0], y_cex[1],
        marker="*", s=230, color="tab:red", edgecolor="black", linewidth=0.5,
        zorder=6, label=r"one-step output $f(x)$",
    )
    ax.annotate(
        "",
        xy=(y_cex[0], y_cex[1]),
        xytext=(x_cex[0], x_cex[1]),
        arrowprops=dict(arrowstyle="->", color="black", linewidth=2.0),
        zorder=4,
    )

    q_min = min(x_cex[0], y_cex[0])
    q_max = max(x_cex[0], y_cex[0])
    v_min = min(x_cex[1], y_cex[1], safe_lower[1])
    v_max = max(x_cex[1], y_cex[1], input_upper[1])
    ax.set_xlim(q_min - 0.22, q_max + 0.22)
    ax.set_ylim(v_min - 0.035, v_max + 0.055)
    ax.set(
        xlabel="state coordinate $q$",
        ylabel="state coordinate $v$",
        title="Counterexample one-step transition",
    )
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


In [ ]:
import torch
import torch.nn as nn

from abcrown import (
    ABCrownSolver,
    ConfigBuilder,
    IOConstraints,
    input_vars,
    output_vars,
)

torch.manual_seed(0)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

DEVICE = "cuda"


## Van der Pol Oscillator

We use the Van der Pol dynamics as an example of a general nonlinear function.
The map contains **no neural network layers**.

For $\mu=1$, one Euler step with step size $0.1$ is

$$
q^+=q+0.1v,\qquad
v^+=v+0.1\big((1-q^2)v-q\big).
$$

The state is $x=(q,v)$ and the one-step map is $f(x)=(q^+,v^+)$.


In [ ]:
plot_vanderpol_trajectories()

## Example 1. Compute Bounds

Goal: compute lower and upper bounds for a nonlinear function on an input set.
For $y=f(x)$ and $x\in\mathcal{X}$, `compute_bounds(...)` returns
bounds for selected output components or linear expressions of $y$.


### Step 1. Implement the Nonlinear Function as an `nn.Module`

`VanDerPolStep` implements the one-step map $f(q,v)=(q^+,v^+)$ as an
`nn.Module`. The verifier analyzes the tensor operations executed in
`forward`.

In [ ]:
class VanDerPolStep(nn.Module):
    def forward(self, state):
        # state has shape (batch, 2), with columns q and v.
        # Slices keep shape (batch, 1), which makes concatenation explicit.
        q = state[:, 0:1]
        v = state[:, 1:2]

        # One explicit Euler step for the Van der Pol oscillator with dt=0.1.
        q_next = q + 0.1 * v
        v_next = v + 0.1 * ((1.0 - q.square()) * v - q)

        # The verifier sees a 2D output y=(q_next, v_next).
        return torch.cat((q_next, v_next), dim=1)

# Move the module to the GPU and switch to evaluation mode.
model = VanDerPolStep().to(DEVICE).eval()

#### Supported PyTorch Operators

`nn.Module.forward` may compose the following supported PyTorch operator families.

| Category | Operators |
|---|---|
| Element-wise arithmetic and math | `add`, `sub`, `mul`, `div`; `neg`, `pow`, `abs`, `square`, `reciprocal`, `exp`, `log`, `sqrt`; `sin`, `cos`, `tan`, `atan` |
| Nonlinear activations | `relu`, `hardtanh`; `tanh`, `sigmoid`, `gelu` |
| Reduction | `reduce sum`, `reduce mean`, `reduce max`; `max`, `min` |
| Linear / Matrix | `linear`; `matmul` |
| Convolutional | `conv2d`; `conv_transpose2d` |
| Shape manipulation | `reshape`, `flatten`; `transpose`; `squeeze`, `unsqueeze`; `expand`; `concat`, `slice`; `gather` |
| Stochastic / Regularization | `dropout`; `softmax` |

Operator support is based on linear relaxations. A nonlinear operator can be
added to verifier support once valid affine lower and upper bounds are
available on each input interval.

<p align="center"><img src="https://github.com/Verified-Intelligence/abCROWN_Control_Tutorial/blob/main/figures/relaxation_demo.png?raw=1" alt="Linear relaxation examples" width="92%" /></p>


### Step 2. Create Symbolic Variables and Solver
### Symbolic Inputs and Outputs

`input_vars(2)` and `output_vars(2)` create symbolic vectors matching the
two input and two output coordinates of `model`. Their elements support
indexing, arithmetic, and comparisons.


In [ ]:
# x represents the 2D input state (q, v).
# y represents the 2D output state (q_next, v_next).
x = input_vars(2)
y = output_vars(2)

#### Solver Configuration

`ConfigBuilder.from_defaults()` starts from the standard verifier settings.
Setting `bab/max_iterations` to `1` makes `compute_bounds(...)` return
a fast CROWN bound.

In [ ]:
# Start from default verifier settings.
# The one-iteration setting returns the initial CROWN bound quickly.
config = ConfigBuilder.from_defaults().set("bab/max_iterations", 1)

# The solver binds together the nonlinear model, symbolic inputs, symbolic
# outputs, and configuration. Constraints are supplied to later computations.
solver = ABCrownSolver(model, x, y, config=config)

### Step 3. Specify Constraints and Objectives
### `IOConstraints`

`IOConstraints` stores symbolic constraints on the input and, when needed,
the output. Bound computation requires an input domain but no output
constraint. Tensor comparisons define an input box directly:

```python
(x >= lower) & (x <= upper)
```

Individual coordinates can also be constrained with expressions such as
`(x[0] >= -3.0) & (x[1] <= -0.45)`. Arithmetic, comparisons, parentheses,
`&`, and `|` produce symbolic constraint expressions.


In [ ]:
# Numerical lower and upper corners of the input box X.
# IOConstraints accepts ordinary CPU tensors for input bounds.
x_lower = torch.tensor([-3.0, -0.55])
x_upper = torch.tensor([3.0, -0.45])

# IOConstraints stores symbolic formulas. For bound computation, only the
# input_constraint is needed: x_lower <= x <= x_upper.
domain = IOConstraints(
    input_vars=x,
    input_constraint=(x >= x_lower) & (x <= x_upper),
)

#### Flexible Constraint Expressions

Symbolic constraints follow standard Python expression syntax. Vector
comparisons, coordinate-wise scalar comparisons, conjunctions, disjunctions,
and linear output formulas can be combined directly.


In [ ]:
# Vector comparison: a compact box constraint x_lower <= x <= x_upper.
box_constraint = (x >= x_lower) & (x <= x_upper)

# Coordinate-wise constraints can be mixed with vector constraints.
velocity_strip = (x[1] >= -0.53) & (x[1] <= -0.47)

# Disjunctions describe unions of symbolic regions.
left_or_right = (
    ((x[0] <= -1.0) & velocity_strip)
    | ((x[0] >= 1.0) & velocity_strip)
)

# Output properties use the same syntax, now over y=(q_next, v_next).
safe_output_box = (
    (y[0] > -3.1) & (y[0] < 3.0)
    & (y[1] > -0.7) & (y[1] < 0.25)
)

# Linear expressions can appear inside comparisons.
tilted_output_halfspace = y[0] + 0.25 * y[1] < 3.05

# IOConstraints can contain both input and output formulas.
demo_property = IOConstraints(
    input_vars=x,
    output_vars=y,
    input_constraint=box_constraint & velocity_strip,
    output_constraint=safe_output_box & tilted_output_halfspace,
)

#### Objectives

The complete output vector `y`, one coordinate such as `y[1]`, or a list of
linear expressions can be passed as the objective.


In [ ]:
# The full output vector asks for componentwise bounds on q_next and v_next.
vector_objective = y

# A scalar objective asks for bounds on one coordinate.
scalar_objective = y[1]

# A list gives one pair of lower/upper bounds for each linear expression.
linear_objectives = [y[0], y[1], y[0] + 0.5 * y[1]]

### Step 4. Run `compute_bounds(...)`

`constraints` specifies the input set and `objective` specifies the
quantities to bound. Passing `y` computes bounds for both $q^+$ and $v^+$.

In [ ]:
# Compute lower and upper bounds on both output coordinates.
bounds = solver.compute_bounds(constraints=domain, objective=y)

# For every x in the input box, bounds.lower <= model(x) <= bounds.upper.
print("lower:", bounds.lower)
print("upper:", bounds.upper)

#### Reachable-Set Bound Visualization

The one-step reachable set is

$$
f(\mathcal{X})=\{f(x):x\in\mathcal{X}\}.
$$

Sampled outputs show the shape of $f(\mathcal{X})$. The output bound rectangle
contains every one-step successor, including unsampled states.


In [ ]:
# Numerical samples are only for visualization; they do not prove bounds.
sample_inputs, sample_outputs = sample_one_step(model, x_lower, x_upper)

# Draw sampled one-step outputs together with the output bound box.
plot_reachable_box(sample_outputs, bounds.lower, bounds.upper)

## Additional Features for Compute Bounds
### Input Branch-and-Bound

Input BaB splits the input domain and recomputes sound bounds on
subdomains. Larger `bab/max_iterations` values allow more splitting.

`bab/timeout` can also impose a wall-clock limit:

```python
ConfigBuilder.from_defaults().set("bab/timeout", 2.0)
```

In [ ]:
# Compare how the bounds change as the Input BaB budget increases.
iteration_limits = [1, 2, 4, 8]
bound_results = []

for limit in iteration_limits:
    config = ConfigBuilder.from_defaults().set("bab/max_iterations", limit)
    limited_solver = ABCrownSolver(model, x, y, config=config)
    q_bounds, v_bounds = [
        limited_solver.compute_bounds(constraints=domain, objective=[coord])
        for coord in (y[0], y[1])
    ]
    bound_results.append({
        key: torch.stack((getattr(q_bounds, key)[0], getattr(v_bounds, key)[0]))
        for key in ("lower", "upper")
    })

v_lower_bounds = [float(result["lower"][1]) for result in bound_results]
sampled_minimum = float(sample_outputs[:, 1].min())
plot_lower_convergence(iteration_limits, v_lower_bounds, sampled_minimum)


### Reachable-Set Bounds with More BaB Iterations

Input BaB combines the bounds from its subdomains into one sound interval
for each output coordinate. The plot compares the initial bound box with
the box obtained after a larger splitting budget.

In [ ]:
# Compare the first and last reachable-set bound boxes.
first_bounds = bound_results[0]
tight_bounds = bound_results[-1]

print("1 iteration:", first_bounds["lower"], first_bounds["upper"])
print("8 iterations:", tight_bounds["lower"], tight_bounds["upper"])

# The smaller aggregate box is obtained by combining BaB subdomain bounds.
plot_bound_comparison(
    sample_outputs,
    first_bounds["lower"], first_bounds["upper"],
    tight_bounds["lower"], tight_bounds["upper"],
)

### Linear Reachable-Set Bounds

Reachable-set bounds are not limited to boxes. Passing several linear
objectives gives bounds in several output directions, and their intersection
forms a polygon in the $(q^+, v^+)$ plane.

$$
\ell_i \le c_i^\top y \le u_i,\qquad
c_i \in \{(1,0),(0,1),(0.2,1),(-0.2,1)\}.
$$


In [ ]:
# Bound the reachable set in four output directions.
linear_directions = [
    (1.0, 0.0),     # q_next
    (0.0, 1.0),     # v_next
    (0.2, 1.0),     # tilted upper-left/lower-right direction
    (-0.2, 1.0),    # tilted lower-left/upper-right direction
]
linear_objectives = [a * y[0] + b * y[1] for a, b in linear_directions]

linear_reachable_bounds = solver.compute_bounds(
    constraints=domain,
    objective=linear_objectives,
)

print("linear lower:", linear_reachable_bounds.lower)
print("linear upper:", linear_reachable_bounds.upper)

plot_linear_reachable_set(
    sample_outputs,
    linear_directions,
    linear_reachable_bounds.lower,
    linear_reachable_bounds.upper,
)


### Returning Linear Bounds

`compute_bounds(..., return_linear_bounds=True)` returns input-dependent
affine functions

$$
A^{\mathrm L}x+b^{\mathrm L}
\leq v^+(x)\leq
A^{\mathrm U}x+b^{\mathrm U}.
$$

The coefficient arrays are stored in `result.linear_bounds`. With Input BaB,
the same result also contains `linear_bounds["subdomains"]`, an affine record
list on split domains. The global affine bounds and the subdomain affine
records are returned together by `compute_bounds(...)`.

In [ ]:
# Use a moderate BaB budget to return both global and subdomain affine data.
linear_config = ConfigBuilder.from_defaults().set("bab/max_iterations", 4)
linear_solver = ABCrownSolver(model, x, y, config=linear_config)

# return_linear_bounds=True exposes global affine bounds and BaB subdomain records.
linear_result = linear_solver.compute_bounds(
    constraints=domain,
    objective=[y[1]],
    return_linear_bounds=True,
)
linear_bounds = linear_result.linear_bounds

# Extract the returned affine coefficients and subdomain records explicitly.
lower_A = linear_bounds["lower_A"]
lower_bias = linear_bounds["lower_bias"]
upper_A = linear_bounds["upper_A"]
upper_bias = linear_bounds["upper_bias"]
lower_subdomain_records = linear_bounds["subdomains"]["lower"]


#### Affine Bound Visualization

The figure compares the global affine lower and upper surfaces with the
piecewise-affine lower surfaces returned on Input BaB subdomains.

In [ ]:
# Global affine surfaces and BaB subdomain surfaces can be visualized directly.
plot_linear_bound_comparison(
    model, x_lower, x_upper,
    lower_A, lower_bias, upper_A, upper_bias, lower_subdomain_records,
)

## Example 2. Satisfiability Verification

Goal: prove a property of the nonlinear function over the input box

$$
\mathcal{X}=\{(q,v):-3.0\le q\le3.0,\;-0.55\le v\le-0.45\}.
$$

The safe output set is

$$
\mathcal{Y}_{\mathrm{safe}}
=\{(q^+,v^+):-3.1<q^+<3.0,\;-0.7<v^+<0.25\}.
$$

The verification property is the implication

$$
x\in\mathcal{X}\quad\Longrightarrow\quad f(x)\in\mathcal{Y}_{\mathrm{safe}}.
$$

`verify(...)` proves this implication and returns a `SolveResult` with
`status="verified"`.

In [ ]:
# Display the nonlinear function outputs and the safe output set.
visual_safe_lower = torch.tensor([-3.1, -0.7])
visual_safe_upper = torch.tensor([3.0, 0.25])
plot_verification_region(sample_outputs, visual_safe_lower, visual_safe_upper)


### Step 1. Use the Van der Pol `nn.Module`

`model` represents the map $f(x)=(q^+,v^+)$. The verifier analyzes the tensor operations in `model.forward`.

### Step 2. Use Symbolic Variables and Create a Solver

`x` and `y` match the input and output coordinates of `model`. The solver
uses the default verifier configuration for the true safety property.

In [ ]:
# x and y are symbolic coordinates for the verifier.
verification_solver = ABCrownSolver(model, x, y, config=ConfigBuilder.from_defaults())

### Step 3. Specify Input and Output Constraints

`IOConstraints` combines the input domain and the output formula. The true
property requires every one-step successor to remain in the safe output
rectangle.

In [ ]:
# Safe output rectangle for y=(q_next, v_next).
q_safe_lower, v_safe_lower = -3.1, -0.7
q_safe_upper, v_safe_upper = 3.0, 0.25
safe_lower = torch.tensor([q_safe_lower, v_safe_lower])
safe_upper = torch.tensor([q_safe_upper, v_safe_upper])

# A true safety property: every output must stay inside the safe rectangle.
safe_property = IOConstraints(
    input_vars=x,
    output_vars=y,
    input_constraint=(x >= x_lower) & (x <= x_upper),
    output_constraint=(
        (y[0] > q_safe_lower) & (y[0] < q_safe_upper)
        & (y[1] > v_safe_lower) & (y[1] < v_safe_upper)
    ),
)

### Step 4. Run `verify(...)`

`verify(...)` returns a `SolveResult` with `status`:

- `verified`: the property was proved;
- `falsified`: a valid violation was found;
- `unknown`: the configured resource budget was inconclusive.

The one-iteration CROWN box crosses the lower boundary of the safe output
set. The Input BaB box lies inside the safe output set, matching the
`verified` result.

In [ ]:
# Prove the true safety property.
safe_verification = verification_solver.verify(constraints=safe_property)

print("safe property:", safe_verification.status)

# Display the initial CROWN and tightened Input BaB output boxes.
plot_verification_bounds(sample_outputs, safe_lower, safe_upper)


### Falsification and Counterexamples

A stricter property changes the lower requirement from $v^+>-0.7$ to
$v^+>-0.55$:

$$
\mathcal{Y}_{\mathrm{strict}}
=\{(q^+,v^+):-3.1<q^+<3.0,\;-0.55<v^+<0.25\}.
$$

`verify(...)` returns `status="falsified"` when it finds a valid input
$x\in\mathcal{X}$ whose output violates the requested property.
`false_result.stats["attack_examples"]` stores inputs returned by the
falsification search.

The plotted transition starts from $x\in\mathcal{X}$ and ends at
$f(x)\notin\mathcal{Y}_{\mathrm{strict}}$.

In [ ]:
# A false property: the required lower bound on v_next is too large.
false_v_lower = -0.55
false_property = IOConstraints(
    input_vars=x,
    output_vars=y,
    input_constraint=(x >= x_lower) & (x <= x_upper),
    output_constraint=(
        (y[0] > q_safe_lower) & (y[0] < q_safe_upper)
        & (y[1] > false_v_lower) & (y[1] < v_safe_upper)
    ),
)

# Falsify the tightened output property.
false_result = verification_solver.verify(constraints=false_property)

# Inputs returned by the falsification search are stored in the SolveResult stats.
candidate_inputs = false_result.stats["attack_examples"]

print("false property:", false_result.status)

# Display x and f(x) in the same state plane.
false_lower = torch.tensor([q_safe_lower, false_v_lower])
plot_counterexample(candidate_inputs, model, x_lower, x_upper, false_lower, safe_upper)